<a href="https://colab.research.google.com/github/myoungjinahn/Machine-Learning-project-team1/blob/main/(1%EC%B5%9C%EC%A2%85)%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A4%80%EB%B9%84_%EA%B3%BC%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2013년 ~ 2025년 9월 ~ 11월 온도 데이터

1. 파일 읽기

- 2013년 9월부터 2025년 11월까지 파일을 업로드



In [5]:
import pandas as pd
from google.colab import files
import io

# 1. 엑셀 파일 직접 업로드
print("기상청에서 다운로드한 엑셀(.xlsx) 파일을 업로드해주세요.")
uploaded = files.upload()

# 업로드된 파일명 확인
file_name = list(uploaded.keys())[0]

try:
    # 2. 데이터 불러오기
    # E9부터 수치가 시작되므로, 8행(제목줄)을 포함하기 위해 상위 7개 행을 건너뜁니다.
    df = pd.read_excel(io.BytesIO(uploaded[file_name]), skiprows=7)

    # 컬럼명 양끝 공백 제거 (인식 오류 방지)
    df.columns = df.columns.str.strip()

    # 3. 날짜 형식 변환 및 필터링을 위한 임시 처리
    df['날짜'] = pd.to_datetime(df['날짜'])

    # 4. 조건 필터링: 2013년~2025년 / 9월, 10월, 11월
    filtered_df = df[
        (df['날짜'].dt.year >= 2013) &
        (df['날짜'].dt.year <= 2025) &
        (df['날짜'].dt.month.isin([9, 10, 11]))
    ].copy()

    # 5. 날짜 포맷 변경 (시간 제거: YYYY-MM-DD 형식)
    filtered_df['날짜'] = filtered_df['날짜'].dt.strftime('%Y-%m-%d')

    # 6. 필요한 컬럼만 추출 (날짜와 최고기온)
    # 기상청 표준 명칭인 '최고기온(℃)'을 사용합니다.
    result = filtered_df[['날짜', '최고기온(℃)']]

    # 7. 엑셀 파일로 저장 및 다운로드
    output_filename = 'filtered_autumn_temperature.xlsx'
    result.to_excel(output_filename, index=False)

    print(f"\n--- 작업 완료 ---")
    print(f"추출된 데이터 개수: {len(result)}개")
    print(f"결과 파일: {output_filename}")

    # 파일 다운로드 실행
    files.download(output_filename)

except Exception as e:
    print(f"\n오류 발생: {e}")
    print("팁: 만약 '최고기온(℃)' 컬럼을 찾지 못한다면, 엑셀 8행에 적힌 정확한 이름을 확인해 보세요.")

기상청에서 다운로드한 엑셀(.xlsx) 파일을 업로드해주세요.


Saving 초기 데이터).2013-2025년 온도 데이터.xlsx to 초기 데이터).2013-2025년 온도 데이터.xlsx

--- 작업 완료 ---
추출된 데이터 개수: 1183개
결과 파일: filtered_autumn_temperature.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 2013년 ~ 2025년 9월 ~ 11월 초미세먼지 데이터

1. 서울특별시 대기환경정보에서 1달씩 엑셀을 뽑아서 특정 데이터만 뽑음

# 2013년 9월 데이터 만들기
- 계절성 평균법 (Seasonal Mean) 이용

In [6]:
import pandas as pd
import numpy as np
from google.colab import files

# 1. 파일 업로드 단계
print("수정할 엑셀 파일(.xlsx)을 업로드해주세요.")
uploaded = files.upload()

# 업로드된 파일명 추출
file_name = list(uploaded.keys())[0]

# 2. 엑셀 데이터 로드
# 엑셀 파일은 pd.read_excel을 사용하며, 앞서 확인한 대로 상단 3줄은 건너뜀
# 'Sheet1' 시트를 기준으로 불러옴
df = pd.read_excel(file_name, sheet_name='Sheet1', skiprows=3)

# 3. 컬럼명 정리
# 첫 번째와 두 번째 컬럼을 제외한 나머지는 1일~31일 데이터임
day_cols = [f'{i}일' for i in range(1, 32)]
# 컬럼 개수에 맞춰 이름을 재설정
df.columns = ['구분', '연월'] + day_cols[:len(df.columns)-2]

# 4. 데이터 타입 변환 (문자열로 되어 있을 수 있는 값을 숫자로 변환)
for col in day_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. 계절성 평균(Seasonal Mean) 계산
# '09월'이 포함된 모든 연도의 데이터를 가져와 일별 평균을 구함
sept_data_all = df[df['연월'].str.contains('09월', na=False)]
sept_mean = sept_data_all[day_cols].mean()

# 6. 2013년 09월 행에 평균값 채워넣기
target_condition = df['연월'].str.contains('2013년09월', na=False)

if target_condition.any():
    # 해당 날짜 컬럼들에 계산된 평균값 대입
    df.loc[target_condition, sept_mean.index] = sept_mean.values
    print(f"\n 2013년 09월 빈칸을 역대 9월 평균치로 채웠습니다.")
else:
    print("\n '2013년09월' 행을 찾을 수 없습니다. 데이터의 '연월' 표기를 확인해주세요.")

# 7. 결과 확인 및 저장
df[day_cols] = df[day_cols].round(1) # 소수점 첫째자리까지 반올림
print("\n--- 보정된 데이터 상단 확인 ---")
print(df.head())

# 수정된 내용을 새로운 엑셀 파일로 저장
output_name = 'fixed_seasonal_data.xlsx'
df.to_excel(output_name, index=False)
print(f"\n 보정 완료! '{output_name}' 파일이 생성되었습니다.")

# 파일 자동 다운로드 (선택 사항)
files.download(output_name)

수정할 엑셀 파일(.xlsx)을 업로드해주세요.


Saving 초기 데이터). 2013~2025년 초미세먼지 가을.xlsx to 초기 데이터). 2013~2025년 초미세먼지 가을.xlsx

 '2013년09월' 행을 찾을 수 없습니다. 데이터의 '연월' 표기를 확인해주세요.

--- 보정된 데이터 상단 확인 ---
   구분        연월  1일  2일  3일  4일  5일  6일  7일  8일  ...  22일  23일  24일  25일  26일  \
0 NaN  2014년10월  13  18  14  10   9  10  16  19  ...   10   16   23   25   20   
1 NaN  2014년11월  21  29  12  13  17  27  19  29  ...   25   34   35   17   16   
2 NaN  2015년09월  16  16  15  20  19   7   7   7  ...   25   25   22   22   28   
3 NaN  2015년10월   8  10  17  12  14  16  23  27  ...   48   29   21   16   16   
4 NaN  2015년11월  24  29  42  55  52  32   8   3  ...   20    5   15    9   24   

   27일  28일  29일  30일   31일  
0   14   24   30   34  24.0  
1   25   16   11    9   NaN  
2   15   14   10    8   NaN  
3   13   13   12   10  16.0  
4   26   28   29   29   NaN  

[5 rows x 33 columns]

 보정 완료! 'fixed_seasonal_data.xlsx' 파일이 생성되었습니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>